# Week 97, Day 3: System design

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w097d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
users, per_day = 10_000_000, 5
print(users * per_day / 86_400, users * per_day / (24 * 60))

> **Key idea.** Lead the forty-five minutes in stages: requirements, estimates, API, data model, high-level design, deep dive, trade-offs. Per second = per day / 86,400, times a peak factor. Storage = items × bytes × days × replication. Servers = peak / capacity at 70%, at least two. Cache = hot fraction × items × bytes. Random code collisions follow the birthday problem: keep the space far larger than the codes. State each trade-off and choose.

### Exercise: Traffic and storage

Write `capacity(users, actions_per_user, peak_factor, bytes_per_action, days, replication=3)`: return a dict with `avg_qps` (actions a day over 86,400, to 1 decimal place), `peak_qps` (average times the peak factor, rounded to a whole number) and `storage_gb` (actions a day times bytes times days times replication, in gigabytes of 10^9 bytes, to 1 decimal place).

In [ ]:
def capacity(users, actions_per_user, peak_factor, bytes_per_action, days, replication=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_capacity():\n    \"aloud, then exactly\"\n    assert capacity(10_000_000, 5, 3, 500, 365) == {\"avg_qps\": 578.7, \"peak_qps\": 1736, \"storage_gb\": 27375.0}\n    assert capacity(1000, 1, 2, 1000, 1, replication=1) == {\"avg_qps\": 0.0, \"peak_qps\": 0, \"storage_gb\": 0.0}")

### Exercise: How many servers?

Write `servers_needed(peak_qps, per_server, headroom=0.3)`: the smallest whole number of servers that handles `peak_qps` with each running at no more than `1 - headroom` of `per_server`, and never fewer than 2.

In [ ]:
import math

def servers_needed(peak_qps, per_server, headroom=0.3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_servers():\n    \"headroom and redundancy\"\n    assert servers_needed(1736, 500) == 5 and servers_needed(10_000, 2_000) == 8\n    assert servers_needed(10, 1000) == 2 and servers_needed(1400, 1000) == 2 and servers_needed(1401, 1000) == 3\n    assert servers_needed(900, 1000, headroom=0) == 2")

### Exercise: Keeping to time

Write `stage_report(notes, end)`: `notes` is a list of `(minute, stage)` saying when each stage started, and `end` is the minute the interview ended. The stages are `requirements`, `estimates`, `api`, `data model`, `high level`, `deep dive` and `trade-offs`. Return lines: `"STAGE: N min"` for each stage visited, in the order visited (each lasts until the next note or `end`); then `"missing: A, B"` for stages never visited (in the standard order), if any; then `"warning: requirements under 3 min"` if requirements took less than 3 minutes (or never happened) and `"warning: out of order"` if the visited stages aren't in the standard order.

In [ ]:
STAGES = ["requirements", "estimates", "api", "data model", "high level", "deep dive", "trade-offs"]

def stage_report(notes, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_good():\n    \"on time, in order\"\n    notes = [(0, \"requirements\"), (5, \"estimates\"), (10, \"api\"), (15, \"data model\"), (20, \"high level\"), (30, \"deep dive\"), (40, \"trade-offs\")]\n    assert stage_report(notes, 45) == [\"requirements: 5 min\", \"estimates: 5 min\", \"api: 5 min\", \"data model: 5 min\",\n                                       \"high level: 10 min\", \"deep dive: 10 min\", \"trade-offs: 5 min\"]\n\ndef test_rushed():\n    \"straight to the boxes\"\n    notes = [(0, \"requirements\"), (1, \"high level\"), (25, \"api\"), (32, \"deep dive\")]\n    assert stage_report(notes, 45) == [\"requirements: 1 min\", \"high level: 24 min\", \"api: 7 min\", \"deep dive: 13 min\",\n                                       \"missing: estimates, data model, trade-offs\", \"warning: requirements under 3 min\",\n                                       \"warning: out of order\"]")

### Exercise: How long should codes be?

Write `collision_chance(n, length, alphabet=62)`: the birthday approximation `1 - exp(-n² / (2N))` with `N = alphabet ** length`, and `code_length(n, max_chance=0.001, alphabet=62)`: the shortest length whose collision chance for `n` codes is at most `max_chance`.

In [ ]:
import math

def collision_chance(n, length, alphabet=62):
    ___

def code_length(n, max_chance=0.001, alphabet=62):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\n\ndef test_chance():\n    \"the birthday problem\"\n    assert math.isclose(collision_chance(23, 1, alphabet=365), 1 - math.exp(-23 * 23 / 730))\n    assert collision_chance(1_000_000, 6) > 0.5 and collision_chance(1_000_000, 8) < 0.003\n    assert collision_chance(0, 4) == 0\n\ndef test_length():\n    \"far larger than the codes\"\n    assert code_length(1_000_000) == 9 and code_length(1_000) == 5 and code_length(10) == 3\n    assert code_length(100_000_000, max_chance=0.01) == 10")

### Exercise: Sizing the cache

Write `cache_gb(items, item_bytes, hot_fraction=0.2, overhead=1.5)`: the memory for the hot fraction of items, multiplied by an overhead for the cache's own structures, in gigabytes (10^9 bytes) rounded to 1 decimal place.

In [ ]:
def cache_gb(items, item_bytes, hot_fraction=0.2, overhead=1.5):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"the hot 20%\"\n    assert cache_gb(500_000_000, 500) == 75.0 and cache_gb(1_000_000, 1_000, hot_fraction=0.1, overhead=1) == 0.1")

### Bug hunt: Minutes, not seconds

A day has 86,400 seconds, but this divides by the minutes in a day and prints `34722.2`. Fix the unit so it prints `578.7`.

In [ ]:
per_day = 10_000_000 * 5
print(round(per_day / (24 * 60), 1))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"86,400\"\n    expect_output(\"578.7\")")

### Bug hunt: Storage on one copy

Every item is stored on three replicas, but this sizes the storage for one and prints `9125.0`. Multiply by the replication factor so it prints `27375.0`.

In [ ]:
per_day, bytes_each, days, replication = 50_000_000, 500, 365, 3
print(round(per_day * bytes_each * days / 1e9, 1))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"three copies\"\n    expect_output(\"27375.0\")")

### Puzzle: Servers with headroom

Arrange `servers_needed`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # seconds in a day
q = ___   # 62 ** 2
r = ___   # servers_needed(10, 1000): the minimum, for redundancy
s = ___   # True or False: start drawing boxes before asking about requirements
t = ___   # the usual replication factor in these estimates

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 86_400 and q == 3844 and r == 2, \"p, q and r\"\n    assert s is False and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a link shortener's numbers

Write `shortener_plan(req)` using today's helpers (given). `req` has `links_per_day`, `reads_per_link_per_day` (reads a day for every link created that day), `bytes_per_link`, `years`, `peak_factor`, `per_server` and `max_chance`. Return lines:

1. `"writes: A/s average, P/s peak"` (A to 1 decimal place, P rounded);
2. `"reads: A/s average, P/s peak"`;
3. `"storage: T TB over Y years (3 copies)"`: links a day × bytes × 365 × years × 3, in terabytes (10^12 bytes), 1 decimal place;
4. `"servers: N for reads"` from `servers_needed`;
5. `"codes: length L for M links"`: M the total links over the years, L from `code_length`;
6. `"cache: G GB"` from `cache_gb` with the total links.

In [ ]:
import math

def servers_needed(peak_qps, per_server, headroom=0.3):
    return max(2, math.ceil(peak_qps / (per_server * (1 - headroom)) - 1e-9))

def collision_chance(n, length, alphabet=62):
    return 1 - math.exp(-n * n / (2 * alphabet ** length))

def code_length(n, max_chance=0.001, alphabet=62):
    length = 1
    while collision_chance(n, length, alphabet) > max_chance:
        length += 1
    return length

def cache_gb(items, item_bytes, hot_fraction=0.2, overhead=1.5):
    return round(items * hot_fraction * item_bytes * overhead / 1e9, 1)

def shortener_plan(req):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("REQ = {\"links_per_day\": 1_000_000, \"reads_per_link_per_day\": 100, \"bytes_per_link\": 500, \"years\": 5,\n       \"peak_factor\": 3, \"per_server\": 2_000, \"max_chance\": 0.001}\n\ndef test_plan():\n    \"the classic first question\"\n    assert shortener_plan(REQ) == [\n        \"writes: 11.6/s average, 35/s peak\",\n        \"reads: 1157.4/s average, 3472/s peak\",\n        \"storage: 2.7 TB over 5 years (3 copies)\",\n        \"servers: 3 for reads\",\n        \"codes: length 12 for 1825000000 links\",\n        \"cache: 273.8 GB\",\n    ]\n\ndef test_small():\n    \"a small one\"\n    lines = shortener_plan({**REQ, \"links_per_day\": 1000, \"years\": 1})\n    assert lines[3] == \"servers: 2 for reads\" and lines[4] == \"codes: length 8 for 365000 links\"")

## Warm-up

A day has 86,400 seconds, not 1,440 (that's minutes). Fifty million actions a day is about 580 a second on average, and a common slip makes it 35,000. Interviewers forgive rough numbers; they notice wrong units.

## Forty-five minutes

A timeline keeps the conversation moving and shows you can lead it:

1. **Requirements** (about 5 minutes): what must it do (functional: shorten a link, redirect), and how well (non-functional: latency, availability, scale, how long links last). Ask; don't assume.
2. **Estimates** (5): traffic, storage, bandwidth, with the arithmetic aloud.
3. **API** (5): the endpoints and their inputs and outputs.
4. **Data model** (5): what's stored, keyed how.
5. **High-level design** (10): the boxes and arrows: clients, load balancer, services, database, cache, queue.
6. **Deep dive** (10): the hardest part, usually chosen with the interviewer: generating unique codes, the hot-key problem, consistency.
7. **Trade-offs and wrap-up** (5): what you'd change at ten times the scale, what could fail, what you'd monitor.

Write the stage on the board as you go; the interviewer can then steer you to whatever they want to hear more about.

## Numbers you can do aloud

- **Requests per second**: actions a day divided by 86,400 (call it 100,000 for mental arithmetic), times a **peak factor** (2 to 10) for the busiest hour.
- **Storage**: items a day times bytes each times days kept, times a **replication factor** (often 3).
- **Servers**: peak requests divided by what one server handles, leaving **headroom** (run at 70% so a spike or a failed server doesn't topple the rest), and never fewer than two.
- **Cache**: the **hot fraction** (often 20% of items get 80% of reads) times the item size.
- **Code length**: random codes from 62 characters (letters and digits) give 62^L possible codes; with n codes in use, the chance two random codes ever collided is about 1 - e^(-n²/2N) (the birthday problem). Choose L so the space is far larger than the codes you'll make.

## Trade-offs

Every choice costs something, and saying what is the point: a cache makes reads fast but can serve stale data; random codes are hard to guess but need a collision check; a counter gives short codes but reveals how many links exist and needs coordination; replication improves availability but writes must reach several copies. "It depends" is only a good answer when followed by "on what, and here's what I'd choose for these requirements".

## Practice

## Quiz

## Daily challenge

## A mock round

With a friend (or aloud to yourself, recorded), take forty-five minutes on **"Design a service that shortens links"**. Note the minute each stage starts and run it through `stage_report` afterwards. Then try **"Design the walk-sharing feature of Waymark for a million users"**, using your capstone as the deep dive.

## Before you go

Tomorrow is a code review interview: you're given someone's code and asked what you'd change, which is Week 96's reviewing, under time.